# 01b — Scrape ICWSM & JCDL Award Data

**ICWSM**: scraped from `icwsm.org/awards/`  
> ⚠️ The official ICWSM awards page only lists up to **2021**. Awards for 2022–2025 are missing from the source.

**JCDL**: scraped live from `jcdl.org/awards.php` — full table including all award types:  
- Vannevar Bush Best Paper Award ← main award of interest  
- Best Student Paper Award  
- Best Short Paper Award  
- Best International Paper Award  
- Best Resource Paper Award  
- Best Poster Award  
- Best Demonstration Award  

All JCDL entries are then matched to OpenAlex via fuzzy title search.

**Output schema** (matches `huang_awards_cleaned.csv`):
```
year | conference | paper_title | paper_url | authors | award_type | award_year
```

Saved to: `../data/raw/icwsm_jcdl_awards_raw.csv`

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, re
from rapidfuzz import fuzz

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"}
SIMILARITY_THRESHOLD = 85

## 1. Scrape ICWSM Awards

Page structure: `<h2>` = award type OR year, `<h3>` = paper title, `<h4>` = authors + original year

> ⚠️ Source only lists up to 2021 — gap is known and documented.

In [2]:
def scrape_icwsm_awards():
    resp = requests.get("https://icwsm.org/awards/", headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    records = []
    current_award_type = None
    current_year = None
    pending_title = None

    AWARD_SECTIONS = {
        "test of time": "Test of Time",
        "best paper": "Best Paper",
        "outstanding paper": "Outstanding Paper",
        "honorable mention": "Honorable Mention",
    }

    for tag in soup.find_all(["h2", "h3", "h4"]):
        text = tag.get_text(separator=" ", strip=True)

        if tag.name == "h2":
            matched = False
            for kw, label in AWARD_SECTIONS.items():
                if re.search(kw, text, re.IGNORECASE):
                    current_award_type = label
                    matched = True
                    break
            if not matched:
                m = re.fullmatch(r"\s*(20\d{2})\s*", text)
                if m:
                    current_year = int(m.group(1))

        elif tag.name == "h3" and current_award_type:
            m = re.fullmatch(r"\s*(20\d{2})\s*", text)
            if m:
                current_year = int(m.group(1))
                pending_title = None
            else:
                pending_title = text

        elif tag.name == "h4" and pending_title and current_year and current_award_type:
            orig_year_m = re.search(r"ICWSM\s*(20\d{2})", text, re.IGNORECASE)
            pub_year = int(orig_year_m.group(1)) if orig_year_m else current_year
            authors = re.sub(r";?\s*ICWSM\s*20\d{2}.*$", "", text, flags=re.IGNORECASE).strip()
            records.append({
                "year": pub_year,
                "conference": "ICWSM",
                "paper_title": pending_title,
                "paper_url": "",
                "authors": authors,
                "award_type": current_award_type,
                "award_year": current_year,
            })
            pending_title = None

    return pd.DataFrame(records)


df_icwsm = scrape_icwsm_awards()
print(f"ICWSM records scraped: {len(df_icwsm)}")
print(f"Years covered: {sorted(df_icwsm['award_year'].unique())}")
df_icwsm

ICWSM records scraped: 37
Years covered: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]


,year,conference,paper_title,paper_url,authors,award_type,award_year
0,2011,ICWSM,Political Polarization on Twitter,,"Michael Connover, Jacob Ratkiewicz, Matthew Fr...",Test of Time,2021
1,2010,ICWSM,Measuring User Influence in Twitter: The Milli...,,"Meeyoung Cha, Hamed Haddadi, Fabricio Benevenu...",Test of Time,2020
2,2009,ICWSM,Gephi: An Open Source Software for Exploring a...,,"Mathieu Bastian, Sebastien Heymann, Mathieu Ja...",Test of Time,2019
3,2007,ICWSM,Personality Impressions Based on Facebook Prof...,,"Samuel D. Gosling, Sam Gaddis, Simine Vazire",Test of Time,2018
4,2018,ICWSM,Wikipedian Self-Governance in Action: Motivati...,,"Ivan Beschastnikh, Travis Kriplean, DavidW. Mc...",Test of Time,2018


## 2. Scrape JCDL Awards

Fetches the full awards table from `jcdl.org/awards.php`.  
The page has one table with a `Year` column and one column per award type.  
We parse every non-empty cell and store the award type from the column header.

Award type mapping:
- `Vannevar Bush Best Paper Award` → `"Vannevar Bush Best Paper"`
- `Best Student Paper Award` → `"Best Student Paper"`
- `Best Short Paper Award` → `"Best Short Paper"`
- `Best International Paper Award` → `"Best International Paper"`
- `Best Resource Paper Award` → `"Best Resource Paper"`
- `Best Poster Award` → `"Best Poster"`
- `Best Demonstration Award` → `"Best Demonstration"`

In [4]:
AWARD_TYPE_MAP = {
    "vannevar": "Vannevar Bush Best Paper",
    "best paper": "Vannevar Bush Best Paper",
    "student": "Best Student Paper",
    "short": "Best Short Paper",
    "international": "Best International Paper",
    "resource": "Best Resource Paper",
    "poster": "Best Poster",
    "demonstration": "Best Demonstration",
    "demo": "Best Demonstration",
}

def normalize_award_type(h2_text):
    h = h2_text.lower().strip()
    for kw, label in AWARD_TYPE_MAP.items():
        if kw in h:
            return label
    return h2_text.strip()


def parse_jcdl_table(table, award_type):
    rows = table.find_all("tr")
    if not rows:
        return []

    # Detect column positions from header row
    header_cells = rows[0].find_all(["th", "td"])
    headers = [c.get_text(strip=True).lower() for c in header_cells]
    year_idx    = next((i for i, h in enumerate(headers) if "year"   in h), 0)
    authors_idx = next((i for i, h in enumerate(headers) if "author" in h), 1)
    title_idx   = next((i for i, h in enumerate(headers) if "title"  in h), 2)

    records = []
    for row in rows[1:]:
        cells = row.find_all(["td", "th"])
        if len(cells) < 2:
            continue
        year_text = cells[year_idx].get_text(strip=True) if year_idx < len(cells) else ""
        m = re.search(r"(\d{4})", year_text)
        if not m:
            continue
        year = int(m.group(1))

        title_cell = cells[title_idx] if title_idx < len(cells) else None
        title = title_cell.get_text(separator=" ", strip=True) if title_cell else ""
        paper_url = ""
        if title_cell:
            link = title_cell.find("a")
            if link and link.get("href"):
                paper_url = link["href"]

        authors_cell = cells[authors_idx] if authors_idx < len(cells) else None
        authors = authors_cell.get_text(separator=", ", strip=True) if authors_cell else ""

        if not title:
            continue

        records.append({
            "year": year,
            "conference": "JCDL",
            "paper_title": title,
            "paper_url": paper_url,
            "authors": authors,
            "award_type": award_type,
            "award_year": year,
        })
    return records


def scrape_jcdl_awards():
    resp = requests.get("https://jcdl.org/awards.php", headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    records = []
    current_award_type = None

    # Walk all top-level elements inside  (or main content div)
    for el in soup.find_all(["h2", "table"]):
        if el.name == "h2":
            current_award_type = normalize_award_type(el.get_text(separator=" ", strip=True))
            print(f"Found section: '{el.get_text(strip=True)}' → '{current_award_type}'")
        elif el.name == "table" and current_award_type:
            rows = parse_jcdl_table(el, current_award_type)
            records.extend(rows)
            current_award_type = None  # reset: each h2 owns exactly one table

    return pd.DataFrame(records)


df_jcdl_raw = scrape_jcdl_awards()
print(f"\nJCDL records scraped: {len(df_jcdl_raw)}")
print(df_jcdl_raw.groupby("award_type").size().reset_index(name="count").to_string(index=False))
df_jcdl_raw

Found section: 'Vannevar Bush Best Paper Award' → 'Vannevar Bush Best Paper'
Found section: 'Best Student Paper Award' → 'Best Student Paper'
Found section: 'Best Short Paper Award' → 'Best Short Paper'
Found section: 'Best International Paper Award' → 'Best International Paper'
Found section: 'Best Resource Paper Award' → 'Best Resource Paper'
Found section: 'Best Poster Award' → 'Best Poster'
Found section: 'Best Demonstration Award' → 'Best Demonstration'

JCDL records scraped: 75
              award_type  count
      Best Demonstration      1
Best International Paper      2
             Best Poster     17
     Best Resource Paper      1
        Best Short Paper      3
      Best Student Paper     22
Vannevar Bush Best Paper     29


## 3. Enrich JCDL with OpenAlex

Queries OpenAlex by title for each JCDL entry.  
Uses `rapidfuzz` to validate — if returned title similarity < 85, match is rejected.  
Adds: `openalex_id`, `openalex_title`, `cited_by_count`, enriched `authors`, `paper_url` (DOI).

In [5]:
def enrich_with_openalex(query_title, mailto="thesis@example.com", threshold=SIMILARITY_THRESHOLD):
    params = {"search": query_title, "per_page": 3, "mailto": mailto}
    resp = requests.get("https://api.openalex.org/works", params=params, timeout=15)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    if not results:
        return None, "", ""

    best, best_score = None, 0
    for r in results:
        r_title = r.get("title") or r.get("display_name") or ""
        score = fuzz.token_sort_ratio(query_title.lower(), r_title.lower())
        if score > best_score:
            best, best_score = r, score

    if best_score < threshold:
        print(f"  SKIPPED (score={best_score:.1f}): best match → '{best.get('title', '')[:70]}'")
        return None, "", ""

    doi_url = f"https://doi.org/{best['doi'].split('doi.org/')[-1]}" if best.get("doi") else ""
    authors_str = "; ".join(
        a["author"]["display_name"] for a in best.get("authorships", []) if a.get("author")
    )
    return best, authors_str, doi_url


enriched_records = []
for _, row in df_jcdl_raw.iterrows():
    title = row["paper_title"]
    try:
        result, authors_str, doi_url = enrich_with_openalex(title)
        rec = row.to_dict()
        rec["openalex_id"]    = result["id"] if result else ""
        rec["openalex_title"] = result.get("title", "") if result else ""
        rec["cited_by_count"] = result["cited_by_count"] if result else None
        if result:
            rec["authors"]   = authors_str
            rec["paper_url"] = doi_url
        status = "OK" if result else "NO MATCH"
        print(f"{row['year']} [{row['award_type']}] {status}: {title[:65]}")
    except Exception as e:
        print(f"{row['year']} ERROR: {e}")
        rec = row.to_dict()
        rec["openalex_id"] = rec["openalex_title"] = ""
        rec["cited_by_count"] = None
    enriched_records.append(rec)
    time.sleep(0.3)

df_jcdl = pd.DataFrame(enriched_records)
matched = df_jcdl["openalex_id"].astype(bool).sum()
print(f"\nTotal JCDL: {len(df_jcdl)} | Matched: {matched} / {len(df_jcdl)}")
df_jcdl[["year", "award_type", "paper_title", "openalex_title", "cited_by_count"]]

Total JCDL: 75 | Matched: 60 / 75


## 4. Combine & Save

Merge ICWSM + JCDL into one clean CSV.  
Dedup key: `(paper_title, year, conference, award_type)` — a paper winning Best Paper AND Test of Time stays as two rows.

In [6]:
COLS = ["year", "conference", "paper_title", "paper_url", "authors", "award_type", "award_year",
        "openalex_id", "openalex_title", "cited_by_count"]

# ICWSM has no OpenAlex columns yet — add empty ones for consistency
for col in ["openalex_id", "openalex_title", "cited_by_count"]:
    if col not in df_icwsm.columns:
        df_icwsm[col] = ""

df_combined = pd.concat([df_icwsm[COLS], df_jcdl[COLS]], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=["paper_title", "year", "conference", "award_type"])
df_combined = df_combined.sort_values(["conference", "award_type", "year"]).reset_index(drop=True)

# Save
import os
os.makedirs("../data/raw", exist_ok=True)
out_path = "../data/raw/icwsm_jcdl_awards_raw.csv"
df_combined.to_csv(out_path, index=False)
print(f"Saved {len(df_combined)} records → {out_path}")

print("\nBreakdown:")
print(df_combined.groupby(["conference", "award_type"]).size().reset_index(name="count").to_string(index=False))

Saved 112 records → ../data/raw/icwsm_jcdl_awards_raw.csv

Breakdown:
conference               award_type  count
     ICWSM               Best Paper     32
     ICWSM             Test of Time      5
      JCDL       Best Demonstration      1
      JCDL Best International Paper      2
      JCDL              Best Poster     17
      JCDL      Best Resource Paper      1
      JCDL         Best Short Paper      3
      JCDL       Best Student Paper     22
      JCDL Vannevar Bush Best Paper     29


## 5. Improved Multi-Stage OpenAlex Matching

The original approach (title search → top-3 → fuzzy threshold 85) left ~15 JCDL entries unmatched and ICWSM fully unenriched.

This section replaces `enrich_with_openalex()` with a 4-stage pipeline:

1. **DOI-first** — if a `paper_url` contains a DOI, hit `api.openalex.org/works/doi:{doi}` directly (deterministic, ~100% accurate)
2. **Wider title search** — `per_page=10` instead of 3, more candidates to score
3. **Author cross-validation** — if fuzzy title score is borderline (70–84), check if ≥1 scraped author appears in OpenAlex result; co-match promotes it to a valid hit
4. **Semantic fallback** — for still-unmatched rows, embed title + top-10 candidates with `sentence-transformers` (all-MiniLM-L6-v2) and pick by cosine similarity ≥ 0.82

Anything surviving all 4 stages unmatched gets `match_status = "manual_review"`.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
from sentence_transformers import SentenceTransformer, util as st_util
import torch

MAILTO = "thesis@example.com"
HARD_THRESHOLD = 85      # title fuzzy score: accept immediately
SOFT_THRESHOLD = 70      # borderline: try author cross-validation
SEMANTIC_THRESHOLD = 0.82  # cosine similarity cutoff for semantic fallback

embedder = SentenceTransformer("all-MiniLM-L6-v2")


def _author_overlap(scraped_authors: str, oa_authorships: list) -> bool:
    """True if any scraped last name appears in any OpenAlex author display_name."""
    scraped_lastnames = {
        name.strip().split()[-1].lower()
        for name in re.split(r"[,;]", scraped_authors)
        if name.strip()
    }
    oa_names = [
        (a.get("author") or {}).get("display_name", "").lower()
        for a in oa_authorships
    ]
    return any(ln in name for ln in scraped_lastnames for name in oa_names if ln)


def _fetch_candidates(query_title: str, n: int = 10) -> list:
    params = {"search": query_title, "per_page": n, "mailto": MAILTO}
    r = requests.get("https://api.openalex.org/works", params=params, timeout=15)
    r.raise_for_status()
    return r.json().get("results", [])


def _fetch_by_doi(doi_url: str) -> dict | None:
    # Extract raw DOI from url
    m = re.search(r"10\.\d{4,}/\S+", doi_url)
    if not m:
        return None
    doi = m.group(0).rstrip("/")
    url = f"https://api.openalex.org/works/doi:{doi}"
    try:
        r = requests.get(url, params={"mailto": MAILTO}, timeout=15)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None


def _pack_result(result: dict) -> tuple:
    """Return (result, authors_str, doi_url)."""
    doi_url = f"https://doi.org/{result['doi'].split('doi.org/')[-1]}" if result.get("doi") else ""
    authors_str = "; ".join(
        a["author"]["display_name"] for a in result.get("authorships", []) if a.get("author")
    )
    return result, authors_str, doi_url


def enrich_v2(query_title: str, scraped_authors: str = "", doi_url: str = "") -> tuple:
    """
    Returns (result_dict | None, authors_str, doi_url, match_stage)
    match_stage: 'doi' | 'title_hard' | 'title_author' | 'semantic' | 'no_match'
    """
    # Stage 1 — DOI direct lookup
    if doi_url:
        hit = _fetch_by_doi(doi_url)
        if hit:
            return (*_pack_result(hit), "doi")

    # Stage 2 + 3 — wider title search + author cross-validation
    candidates = _fetch_candidates(query_title, n=10)
    best, best_score = None, 0
    for c in candidates:
        c_title = c.get("title") or c.get("display_name") or ""
        score = fuzz.token_sort_ratio(query_title.lower(), c_title.lower())
        if score > best_score:
            best, best_score = c, score

    if best_score >= HARD_THRESHOLD:
        return (*_pack_result(best), "title_hard")

    if best_score >= SOFT_THRESHOLD and scraped_authors:
        if _author_overlap(scraped_authors, best.get("authorships", [])):
            return (*_pack_result(best), "title_author")

    # Stage 4 — semantic similarity fallback
    if candidates:
        cand_titles = [c.get("title") or c.get("display_name") or "" for c in candidates]
        q_emb = embedder.encode(query_title, convert_to_tensor=True)
        c_embs = embedder.encode(cand_titles, convert_to_tensor=True)
        sims = st_util.cos_sim(q_emb, c_embs)[0]
        best_idx = int(torch.argmax(sims))
        if float(sims[best_idx]) >= SEMANTIC_THRESHOLD:
            return (*_pack_result(candidates[best_idx]), "semantic")

    return None, "", "", "no_match"

## 6. Re-enrich Both JCDL and ICWSM with `enrich_v2`

Runs the full multi-stage pipeline on all 112 records (37 ICWSM + 75 JCDL).  
Overwrites `../data/raw/icwsm_jcdl_awards_raw.csv` with the enriched output.  
New column `match_stage` tracks how each record was matched.

In [ ]:
def enrich_df(df: pd.DataFrame, label: str) -> pd.DataFrame:
    records = []
    for _, row in df.iterrows():
        title = row["paper_title"]
        authors = row.get("authors", "")
        doi_url = row.get("paper_url", "")
        try:
            result, authors_str, resolved_doi, stage = enrich_v2(title, authors, doi_url)
            rec = row.to_dict()
            rec["openalex_id"]    = result["id"] if result else ""
            rec["openalex_title"] = (result.get("title") or result.get("display_name") or "") if result else ""
            rec["cited_by_count"] = result["cited_by_count"] if result else None
            rec["match_stage"]    = stage
            if result:
                rec["authors"]   = authors_str
                rec["paper_url"] = resolved_doi
            status_icon = {"doi": "🔑", "title_hard": "✅", "title_author": "🤝", "semantic": "🧠", "no_match": "❌"}.get(stage, "?")
            print(f"{status_icon} {row['year']} [{row['award_type'][:25]}] {stage}: {title[:55]}")
        except Exception as e:
            print(f"💥 {row['year']} ERROR: {e}")
            rec = row.to_dict()
            rec["openalex_id"] = rec["openalex_title"] = ""
            rec["cited_by_count"] = None
            rec["match_stage"] = "error"
        records.append(rec)
        time.sleep(0.35)
    df_out = pd.DataFrame(records)
    matched = df_out["openalex_id"].astype(bool).sum()
    print(f"\n{label}: {matched}/{len(df_out)} matched")
    print(df_out["match_stage"].value_counts().to_string())
    return df_out


print("=== JCDL ===")
df_jcdl_v2 = enrich_df(df_jcdl_raw, "JCDL")

print("\n=== ICWSM ===")
df_icwsm_v2 = enrich_df(df_icwsm, "ICWSM")

# Combine & save
COLS_V2 = ["year", "conference", "paper_title", "paper_url", "authors", "award_type", "award_year",
           "openalex_id", "openalex_title", "cited_by_count", "match_stage"]

for col in COLS_V2:
    for df_ in [df_jcdl_v2, df_icwsm_v2]:
        if col not in df_.columns:
            df_[col] = ""

df_final = pd.concat([df_icwsm_v2[COLS_V2], df_jcdl_v2[COLS_V2]], ignore_index=True)
df_final = df_final.drop_duplicates(subset=["paper_title", "year", "conference", "award_type"])
df_final = df_final.sort_values(["conference", "award_type", "year"]).reset_index(drop=True)

out_path = "../data/raw/icwsm_jcdl_awards_raw.csv"
df_final.to_csv(out_path, index=False)
total_matched = df_final["openalex_id"].astype(bool).sum()
print(f"\n✅ Saved {len(df_final)} records → {out_path}")
print(f"Overall match rate: {total_matched}/{len(df_final)} ({100*total_matched/len(df_final):.1f}%)")
print("\nStage breakdown:")
print(df_final["match_stage"].value_counts().to_string())
print("\nManual review needed:")
print(df_final[df_final["match_stage"] == "no_match"][["conference", "year", "award_type", "paper_title"]].to_string(index=False))